### Process Orders Data

![image_1787942150279.png](./image_1787942150279.png "image_1787942150279.png")

In [0]:
import pyspark.sql.functions as F
from pyspark import pipelines as dp

#### 1. Loading orders data from the source system

In [0]:

@dp.table(
    name = 'bronze_orders',
    table_properties = {'quality': 'bronze'},
    comment = 'Raw orders data from source system'
)
def create_bronze_orders():
    return (
        spark
        .readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'json')
        .option('cloudFiles.inferColumnTypes', 'true')
        .load('dbfs:/Volumes/circuitbox/landing/operational_data/orders')
        .select(
            '*',
            F.col('_metadata.file_path').alias('input_file_path'),
            F.current_timestamp().alias('ingestion_timesamp')
        )
    )

####2. Applying DQ and performing transformation logic

In [0]:

@dp.table(
    name = 'silver_orders_clean',
    table_properties = {'quality': 'silver'},
    comment = 'Orders data with schema and quality checks'
)
@dp.expect_or_fail(
    'valid_customer','customer_id IS NOT NULL'
)
@dp.expect_or_fail(
    'valid_order_id','order_id IS NOT NULL'
)
@dp.expect(
    'valid_order_status',
    'order_status IN ("Pending", "Shipped", "Cancelled", "Completed")'
)
@dp.expect(
    'valid_payment_method',
    'payment_method IN ("Credit Card", "PayPal", "Bank Transfer")'
)

def create_silver_orders_clean():
    return (
        spark
        .table('LIVE.bronze_orders')
        .select(
            'customer_id',
            'items',
            'order_id',
            'order_status',
            F.col('order_timestamp').cast('timestamp').alias('order_timestamp'),
            'payment_method'
        )
    )

####3. Explode the items column and write the flatten data in silver layer

In [0]:
@dp.table(
    name = 'silver_orders',
    table_properties = {'quality': 'silver'},
    comment = 'Orders data post exploding items column'
)
def create_silver_orders():
    return(
        spark.readStream
        .table('LIVE.silver_orders_clean')
        .select(
            'customer_id',
            'order_id',
            'order_status',
            'order_timestamp',
            'payment_method',
            F.explode(F.col('items')).alias('item')
        )
        .select(
            'customer_id',
            'order_id',
            'order_status',
            'order_timestamp',
            'payment_method',
            'item.item_id',
            'item.name',
            'item.price',
            'item.quantity',
            'item.category'
        )
    )
